# 15 - accDM daughter: does a universal fluid closure exist?

Decide **A/B/C** (one formula in `x=k/k_fs` / f-dependent coefficients / interpolation tables) **before** fitting anything. We read the exact daughter response pointwise and test whether the effective `ceff2` and `cvis2` sectors each collapse onto one function of `x=k/k_fs` across `(tau, f, eta)`, over `k <= 1 Mpc^-1`, `f_acc <= 0.3`.

Spec: `docs/superpowers/specs/2026-07-05-fluid-closure-feasibility-diagnostic.md`. Plan: `docs/superpowers/plans/2026-07-05-fluid-closure-feasibility-diagnostic.md`. Style follows notebook 7.

**Why this first:** notebook 7 calibrated a single `amp` on `P(k)` (expensive, confounded by the high-k blow-up) and only tuned `ceff2`, leaving `cvis2` at its default. This measures whether a universal closure is even achievable, and whether the `cvis2` sector carries structure the `ceff2`-only fit ignores. Pure Python, no rebuild - all quantities are already emitted by the exact run.

**Quantities read per (k, tau)** (daughter = ncdm index 1): `delta_ncdm[1]`, `theta_ncdm[1]`, `shear_ncdm[1]`, `cs2_ncdm[1]` = delta_p/delta_rho, `w_sigma[1]`/`w_theta[1]` (momentum-weighted effective sound speeds for the sigma/theta moments), `k_fss_acc[1]` = sqrt(3/2) aH / sqrt(ca2).

In [ ]:
import sys; sys.path.insert(0, '.')          # import helpers from notebooks_test/
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from classy import Class
from fluid_closure_helpers import (
    ca2_from_kfs, sound_speed_response, shear_response,
    log_upper_envelope, collapse_band,
)

plt.rcParams.update({
    'mathtext.fontset': 'stix', 'font.family': 'serif', 'font.size': 11,
    'axes.labelsize': 12, 'legend.fontsize': 9, 'lines.linewidth': 1.5, 'figure.dpi': 200})
qual_colors = ['#377eb8', '#ff7f00', '#4daf4a', '#f781bf', '#984ea3']

# --- base cosmology + accDM model (same setup as notebook 7) ---
omega_b, omega_cdm0 = 0.022383, 0.12011
A_s, n_s, tau_reio, H0 = 2.1005829616811546e-9, 0.96605, 0.0543, 67.32
base_params = {'omega_b': omega_b, 'omega_cdm': omega_cdm0, 'H0': H0,
               'A_s': A_s, 'n_s': n_s, 'tau_reio': tau_reio}
PREC = {'output': 'mPk', 'P_k_max_1/Mpc': 10.0, 'z_max_pk': 0.0,
        'evolver': 0, 'reionization_z_start_max': 80}

A_T, MASS, KAPPA = 0.13, 1e16, 6.0
A_REC = 1.0 / (1.0 + 1090.0)

# --- diagnostic sweep grids ---
K_GRID   = np.logspace(-2, 0.0, 18)          # 1/Mpc: sub-horizon, spans below to above k_fs
F_LIST   = [0.05, 0.1, 0.2, 0.3]             # daughter fraction up to the target
ETA_LIST = [0.1, 1.0]                          # production (cold) + one warmer boost

# eps_acc(eta) exactly as input.c precomputes it (for the fit-shape overlay).
def eps_acc_of_eta(eta):
    e2 = eta*eta
    return -e2 + np.sqrt(e2*e2 + 4*e2*eta + 5*e2 + 2*eta) - 2*eta

def accdm_params(eta, f_acc=0.1, kappa=KAPPA):
    """Exact-hierarchy accDM params for a given boost eta and daughter fraction f_acc."""
    ocdm = omega_cdm0 * (1 + f_acc*(1 - A_REC**kappa)/(1 + (A_REC/A_T)**kappa))**(-1)
    p = dict(base_params); p.update(PREC)
    p.update({'omega_cdm': ocdm,
              'vary_Gamma_acc': 'yes', 'kappa_acc': kappa, 'a_t_acc': A_T,
              'f_acc': f_acc, 'eta_acc': eta,
              'm_acc_in_GeV': MASS, 'm_cdm_in_GeV': MASS,
              'N_ncdm': 2, 'deg_ncdm': '3, 1',
              'm_ncdm': '0.02, {:.6e}'.format(MASS*1e9),
              'T_ncdm': '0.71611, 1', 'ncdm_quadrature_strategy': '0, 4',
              'ncdm_N_momentum_bins': '15, 501', 'N_ur': 0.00441,
              'background_Nloga': 5001, 'gauge': 'synchronous',
              'get_perturbations_in_current_gauge': 'yes',
              'ncdm_fluid_trigger_tau_over_tau_k': 25,
              'ncdm_fluid_approximation': 3})          # 3 = none (exact hierarchy)
    return p

print('setup OK; W(eta=0.1) =', round(1 - 2*eps_acc_of_eta(0.1), 4),
      '| grids: k<=1 x{}, f x{}, eta x{}'.format(K_GRID.size, len(F_LIST), len(ETA_LIST)))

## Task 2 - tau-series extractor

Run the exact hierarchy once and keep the **full tau-series** (not just z=0) of the daughter's `{delta, theta, shear, delta_p/delta_rho, w_sigma, w_theta, k_fs}` for each requested k. `aH(tau)` is recovered from the **background** (`a*H` interpolated on conformal time) - *not* from `k_fs*sqrt(cs2)`, since `k_fss_acc` uses the background `ca2` while `cs2_ncdm` is `delta_p/delta_rho`. Base `ca2` then follows from `ca2 = (3/2)(aH/k_fs)^2`.

In [ ]:
def _find_key(d, want):
    if want in d:
        return want
    for kk in d:
        if kk.replace(' ', '').startswith(want.replace(' ', '')):
            return kk
    raise KeyError('{!r} not found; available: {}'.format(want, list(d.keys())))

def extract_daughter_series(params, k_list):
    """Run exact CLASS; return {k: dict of daughter tau-series arrays}."""
    ks = np.sort(np.asarray(k_list, float))
    p = dict(params); p['k_output_values'] = ', '.join('{:.8e}'.format(k) for k in ks)
    M = Class(); M.set(p); M.compute()
    perts = M.get_perturbations()['scalar']            # one dict per k, sorted-k order
    bg = M.get_background()
    tau_bg = np.asarray(bg['conf. time [Mpc]'], float)
    a_bg   = 1.0 / (1.0 + np.asarray(bg['z'], float))
    H_bg   = np.asarray(bg['H [1/Mpc]'], float)
    o = np.argsort(tau_bg); tau_bg, a_bg, H_bg = tau_bg[o], a_bg[o], H_bg[o]
    kd  = _find_key(perts[0], 'delta_ncdm[1]'); kt  = _find_key(perts[0], 'theta_ncdm[1]')
    ksh = _find_key(perts[0], 'shear_ncdm[1]'); kc  = _find_key(perts[0], 'cs2_ncdm[1]')
    kws = _find_key(perts[0], 'w_sigma[1]');    kwt = _find_key(perts[0], 'w_theta[1]')
    kf  = _find_key(perts[0], 'k_fss_acc[1]');  ktau = _find_key(perts[0], 'tau')
    out = {}
    for k, d in zip(ks, perts):
        tau = np.asarray(d[ktau], float)
        aH  = np.interp(tau, tau_bg, a_bg) * np.interp(tau, tau_bg, H_bg)
        out[k] = dict(tau=tau, aH=aH,
                      delta=np.asarray(d[kd], float), theta=np.asarray(d[kt], float),
                      shear=np.asarray(d[ksh], float), dpr=np.asarray(d[kc], float),
                      w_sigma=np.asarray(d[kws], float), w_theta=np.asarray(d[kwt], float),
                      k_fs=np.asarray(d[kf], float))
    M.struct_cleanup(); M.empty()
    return out

In [ ]:
# structure-check: one cheap extraction, fail fast on a bad column name or a bad ca2
_probe = extract_daughter_series(accdm_params(ETA_LIST[0], f_acc=0.1), K_GRID[:3])
assert set(_probe.keys()) == set(K_GRID[:3])
_s = _probe[K_GRID[0]]
for q in ('theta', 'shear', 'dpr', 'w_sigma', 'w_theta', 'k_fs', 'aH', 'tau'):
    assert _s[q].shape == _s['delta'].shape, q
g = (_s['k_fs'] > 0) & np.isfinite(_s['aH'])
ca2 = ca2_from_kfs(_s['k_fs'][g], _s['aH'][g])
print('extractor OK; tau samples per k =', _s['delta'].size,
      '| ca2 range = [{:.2e}, {:.2e}]'.format(ca2.min(), ca2.max()))
assert np.all(ca2 <= 1.0 + 1e-6), 'ca2 > 1 -> aH/k_fs mismatch (check background keys)'

## Task 3 - assemble the response envelopes across (f, eta)

Two normalized effective sound speeds, both directly comparable and both modeled by the fit as `1 + amp*W*sqrt(x)` in the current scheme:
- **ceff2 sector** `R_c = (delta_p/delta_rho) / ca2` (from `cs2_ncdm[1]`).
- **cvis2 sector** `R_s = w_sigma / ca2` (from `w_sigma[1]`, CLASS's shear-moment effective sound speed).

Both oscillate and flip sign above `k_fs`, so we compare on the **upper envelope** of `|.|` (the closure is an envelope, not a pointwise match). `R_v = k*sigma/theta` is also built as a kinematic cross-check for Task 5.

In [ ]:
def build_responses(f, eta):
    ser = extract_daughter_series(accdm_params(eta, f_acc=f), K_GRID)
    xs, rc, rs, rv = [], [], [], []
    for k, s in ser.items():
        g = np.isfinite(s['k_fs']) & (s['k_fs'] > 0) & np.isfinite(s['aH'])
        if not np.any(g):
            continue
        x   = k / s['k_fs'][g]
        ca2 = ca2_from_kfs(s['k_fs'][g], s['aH'][g])
        xs.append(x)
        rc.append(sound_speed_response(s['dpr'][g], ca2))
        rs.append(sound_speed_response(s['w_sigma'][g], ca2))
        rv.append(shear_response(k, s['shear'][g], s['theta'][g]))
    x = np.concatenate(xs); rc = np.concatenate(rc)
    rs = np.concatenate(rs); rv = np.concatenate(rv)
    return {'Rc': log_upper_envelope(x, rc), 'Rs': log_upper_envelope(x, rs),
            'Rv': log_upper_envelope(x, rv)}

RESPONSES = {}
for eta in ETA_LIST:
    for f in tqdm(F_LIST, desc='eta={}'.format(eta)):
        RESPONSES[(f, eta)] = build_responses(f, eta)
print('built', len(RESPONSES), 'responses:', list(RESPONSES.keys()))

## Task 4 - collapse plots + A/B/C decision

Overlay every `(f, eta)` envelope; color by `f`, linestyle by `eta`. Quantify collapse two ways: **pooled over all** `(f, eta)` (does one universal curve fit?) and **within fixed f** (does it collapse once `f` is held?). `TOL = 0.15` = 15% band width at fixed `x`.

In [ ]:
X_EVAL = np.logspace(-1, 3, 60)
W0 = 1 - 2*eps_acc_of_eta(0.1)

def band_all(w):
    return collapse_band(X_EVAL, [RESPONSES[k][w] for k in RESPONSES])[1]
def band_fixed_f(w):
    return float(np.nanmax([collapse_band(X_EVAL, [RESPONSES[(f, e)][w] for e in ETA_LIST])[1]
                            for f in F_LIST]))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), constrained_layout=True)
ls_eta = {ETA_LIST[0]: '-', ETA_LIST[1]: '--'}
col_f  = {f: qual_colors[i] for i, f in enumerate(F_LIST)}
for (f, eta), r in RESPONSES.items():
    for ax, w in zip(axes, ('Rc', 'Rs')):
        x, env = r[w]
        if x.size:
            ax.loglog(x, env, ls_eta[eta], color=col_f[f], alpha=0.85,
                      label='f={}, eta={}'.format(f, eta))
axes[0].loglog(X_EVAL, 1 + 0.2*W0*np.sqrt(X_EVAL), 'k:', label=r'paper fit $1+0.2W\sqrt{x}$')
axes[0].set_title(r'$R_c=(\delta p/\delta\rho)/c_a^2$  (ceff2 sector)')
axes[1].set_title(r'$R_\sigma=w_\sigma/c_a^2$  (cvis2 sector)')
for ax in axes:
    ax.set_xlabel(r'$x=k/k_{\rm fs}$'); ax.grid(True, which='both', alpha=0.3)
axes[0].legend(fontsize=7, ncol=2); plt.show()

band = {w: (band_all(w), band_fixed_f(w)) for w in ('Rc', 'Rs', 'Rv')}
print('band (pooled-all, fixed-f):')
for w in ('Rc', 'Rs', 'Rv'):
    print('  {:>3}: {:.3f}, {:.3f}'.format(w, *band[w]))

In [ ]:
TOL = 0.15
def verdict(w):
    pooled, fixedf = band[w]
    if not np.isfinite(pooled):
        return 'NO DATA'
    if pooled < TOL:
        return 'UNIVERSAL (one formula in x)'
    if np.isfinite(fixedf) and fixedf < TOL:
        return 'F-DEPENDENT (f-varying coefficients)'
    return 'NO COLLAPSE (interpolation tables)'

vc, vs = verdict('Rc'), verdict('Rs')
print('R_c (ceff2 sector):', vc)
print('R_s (cvis2 sector):', vs)

def sev(v):
    return 0 if 'UNIVERSAL' in v else (1 if 'F-DEP' in v else 2)
overall = max(sev(vc), sev(vs))
print('\nAPPROACH ->', ['B/A universal (one formula in x)',
                        'A with f-dependent coefficients',
                        'C interpolation tables'][overall])
if sev(vs) > 0:
    print('cvis2 sector carries structure the current ceff2-only fit ignores -> fit cvis2 too.')
else:
    print('cvis2 sector collapses cleanly -> a single cvis2(x) is enough alongside ceff2(x).')

## Task 5 - cross-checks + feasibility-gate verdict

1. Kinematic shear response `R_v = k*sigma/theta` as an independent probe of the cvis2 sector - if it gives the same verdict as `R_s`, the conclusion is robust to the probe.
2. Measured `R_c` vs the paper (`0.2`) and in-code (`0.25`, `perturbations.c:8865`) fit amplitudes - a quick read on whether the *existing* amplitude is even in the right ballpark, independent of the collapse question.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), constrained_layout=True)
# (left) kinematic shear cross-check
for (f, eta), r in RESPONSES.items():
    x, env = r['Rv']
    if x.size:
        axes[0].loglog(x, env, ls_eta[eta], color=col_f[f], alpha=0.85,
                       label='f={}, eta={}'.format(f, eta))
axes[0].set_title(r'cross-check $R_v=k\sigma/\theta$ (kinematic shear)')
axes[0].set_xlabel(r'$x=k/k_{\rm fs}$'); axes[0].grid(True, which='both', alpha=0.3)
axes[0].legend(fontsize=7, ncol=2)
# (right) measured R_c vs the two fit amplitudes
x0, rc0 = RESPONSES[(0.1, 0.1)]['Rc']
axes[1].loglog(x0, rc0, 'o-', color=qual_colors[0], label='measured $R_c$ (f=0.1, eta=0.1)')
axes[1].loglog(X_EVAL, 1 + 0.20*W0*np.sqrt(X_EVAL), 'k:',  label='paper amp 0.20')
axes[1].loglog(X_EVAL, 1 + 0.25*W0*np.sqrt(X_EVAL), 'r--', label='in-code amp 0.25')
axes[1].set_title(r'$R_c$ vs fit amplitude'); axes[1].set_xlabel(r'$x=k/k_{\rm fs}$')
axes[1].grid(True, which='both', alpha=0.3); axes[1].legend(fontsize=8)
plt.show()

print('R_v band (pooled, fixed-f): {:.3f}, {:.3f}'.format(*band['Rv']))
print('R_v verdict:', verdict('Rv'), '| matches R_s:', verdict('Rv') == vs)

## Verdict

*(Fill the bracketed numbers from the executed `band` dict and printed verdicts.)*

- **R_c (ceff2) collapse:** pooled band = [FILL], fixed-f band = [FILL] -> **[UNIVERSAL / F-DEP / NO-COLLAPSE]**.
- **R_s (cvis2) collapse:** pooled band = [FILL], fixed-f band = [FILL] -> **[UNIVERSAL / F-DEP / NO-COLLAPSE]**; kinematic `R_v` cross-check [agrees / disagrees].
- **Chosen approach:** [B/A universal | A with f-dependent coefficients | C tables].
- **Does cvis2 matter?** `R_s` shows [structure / no structure] the current ceff2-only fit ignores -> [fit cvis2 too / ceff2 alone suffices].
- **Amplitude read:** measured `R_c` sits [above / below / on] both the 0.20 and 0.25 fit amplitudes -> [recalibrate amp / current amp OK].
- **Feasibility gate:** at f=0.3 the converged exact needs `q_size ~5001` (`memory: accdm-fluid-f-boundary`); the fluid is worth pursuing only if the chosen approach reaches ~1% P(k) over k<=1 **cheaper** than exact + the q(f) schedule (notebook 14). If the verdict is C, or the cvis2 sector shows no usable structure, route to notebook 14 instead.
- **Next:** write the follow-up plan (the fit) matched to this verdict. Do NOT touch `perturbations_ceff2_ncdm` before then.

**Keep in sync:** `fluid_closure_helpers.py` is unit-tested in `test_fluid_closure_helpers.py`; the daughter column layout it assumes is `delta/theta/shear/cs2/w_p/w_theta/w_sigma/k_fss_acc` per ncdm species (`perturbations.c:3375-3390`).